# Punjab Paper Figures

This notebook regenerates the Punjab paper figures from saved Punjab data products and saved inversion outputs.

The figure code is defined locally in this notebook so the plotting style, captions, and unit labels can be edited in one place.


In [ ]:
from pathlib import Path
import json

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path('/home/ubuntu/work/punjab')
DATA_DIR = Path('/mnt/data/aoi_punjab')
OUT_DIR = ROOT / 'outputs' / 'punjab_prior'
FIG_DIR = OUT_DIR / 'figures'

VEL_PATH = DATA_DIR / 'vel_ll.h5'
COH_PATH = DATA_DIR / 'coh_ll.h5'
DISP_PATH = DATA_DIR / 'disp_all_ll.h5'
DATE_PATH = DATA_DIR / 'aquisition_dates_ll.h5'
ARCHIVE_PATH = OUT_DIR / 'punjab_phase1_baseline_prediction_archive_expanded_tiles.h5'
S0_NC_PATH = OUT_DIR / 'punjab_phase1_baseline_prediction_expanded_tiles_S0.nc'
SG_NC_PATH = OUT_DIR / 'punjab_phase1_baseline_prediction_expanded_tiles_Sg.nc'
SAMPLE_PIXELS_CSV = OUT_DIR / 'punjab_phase1_baseline_export_sample_pixels.csv'
PAPER_RESULTS_JSON = OUT_DIR / 'punjab_paper_results_summary.json'
EXPANDED_DIAG_JSON = OUT_DIR / 'punjab_phase1_pilot_diagnostics_grouped_support_expanded.json'

FIG5_PATH = FIG_DIR / 'punjab_paper_figure5_support_mask_velocity.png'
FIG6_PATH = FIG_DIR / 'punjab_paper_figure6_baseline_residual_gallery.png'
FIG7_PATH = FIG_DIR / 'punjab_paper_figure7_prior_ablation_comparison.png'
FIG8_PATH = FIG_DIR / 'punjab_paper_figure8_baseline_export_panel.png'

# Edit unit labels here if you want different wording.
VELOCITY_UNIT_LABEL = 'native velocity units (likely mm yr$^{-1}$)'
DEFORMATION_UNIT_LABEL = 'native deformation units'
NORMALIZED_DEFORMATION_UNIT_LABEL = 'normalized deformation units'
STORAGE_UNIT_LABEL = 'relative storage anomaly [model units]'
COHERENCE_UNIT_LABEL = 'coherence [0-1]'
FRACTION_UNIT_LABEL = 'fraction [0-1]'

COHERENCE_THRESHOLD = 0.20
TIME_VALID_FRACTION_THRESHOLD = 0.05
TILE_SIZE = 64
SUPPORT_MARGIN_PIXELS = 64

PHYSICS_CFG = {
    'E': 1e9,
    'nu': 0.25,
    'rho_w': 1000.0,
    'g': 9.81,
    'alpha': 0.8,
    'Hg': 150.0,
    'Seff': 0.2,
    'dx': 10000.0,
    'dy': 10000.0,
    'a_load': 3000.0,
    'a_poro': 3000.0,
}


SYN_FIG1_PATH = ROOT / 'outputs' / 'figures' / 'synthetic_elastic_vs_poro_magnitude.png'
SYN_FIG2_PATH = ROOT / 'outputs' / 'figures' / 'synthetic_two_layer_balanced_dualdec_frequency_vs_baselines.png'
SYN_FIG3_PATH = ROOT / 'outputs' / 'figures' / 'synthetic_two_layer_balanced_conditioned_dualdec_hybrid_sg_vs_baselines.png'
SYN_FIG4_PATH = ROOT / 'outputs' / 'figures' / 'synthetic_two_layer_balanced_conditioned_dualdec_hybrid_sg_multiseed.png'
SYN_CLEAN_SUMMARY_JSON = ROOT / 'outputs' / 'synthetic_two_layer_balanced_dualdec_frequency_summary.json'
SYN_HYBRID_SUMMARY_JSON = ROOT / 'outputs' / 'synthetic_two_layer_balanced_conditioned_dualdec_hybrid_sg_summary.json'
SYN_MULTISEED_SUMMARY_CSV = ROOT / 'outputs' / 'synthetic_two_layer_balanced_conditioned_dualdec_hybrid_sg_multiseed_summary.csv'


## Local Figure Functions

All figure-generation functions are defined below so you can change labels, scales, captions, and layout in one place.


In [ ]:
def parse_acquisition_dates(date_path):
    with h5py.File(date_path, 'r') as handle:
        raw = handle['acquisition_dates'][0]
    return pd.DatetimeIndex(
        pd.to_datetime([value.decode() if isinstance(value, bytes) else str(value) for value in raw], format='%d-%b-%Y')
    )


def load_simple_h5_grid(path, dataset='z'):
    with h5py.File(path, 'r') as handle:
        lat = handle['lat'][:]
        lon = handle['lon'][:]
        grid = handle[dataset][:]
    return lat, lon, grid


def compute_time_valid_fraction(disp_path, dataset='z', chunk_size=12):
    with h5py.File(disp_path, 'r') as handle:
        lat = handle['lat'][:]
        lon = handle['lon'][:]
        disp = handle[dataset]
        n_time = disp.shape[0]
        valid_sum = np.zeros(disp.shape[1:], dtype=np.float32)
        for start in range(0, n_time, chunk_size):
            stop = min(start + chunk_size, n_time)
            valid_sum += np.isfinite(disp[start:stop]).sum(axis=0, dtype=np.float32)
    return lat, lon, valid_sum / float(n_time)


def build_support_mask(coherence, time_valid_fraction, coherence_threshold=COHERENCE_THRESHOLD, time_valid_fraction_threshold=TIME_VALID_FRACTION_THRESHOLD):
    return (
        np.isfinite(coherence)
        & np.isfinite(time_valid_fraction)
        & (coherence >= coherence_threshold)
        & (time_valid_fraction >= time_valid_fraction_threshold)
    )


def map_extent(lat, lon):
    return [float(lon.min()), float(lon.max()), float(lat.min()), float(lat.max())]


def percentile_limits(array, low=2.0, high=98.0, symmetric=False, fallback=1.0):
    finite = array[np.isfinite(array)]
    if finite.size == 0:
        return (-fallback, fallback) if symmetric else (0.0, fallback)
    if symmetric:
        bound = float(np.nanpercentile(np.abs(finite), high))
        bound = bound if bound > 0 else fallback
        return -bound, bound
    lo, hi = np.nanpercentile(finite, [low, high])
    if not np.isfinite(lo):
        lo = 0.0
    if not np.isfinite(hi) or hi == lo:
        hi = lo + fallback
    return float(lo), float(hi)


def load_archive_components(archive_path):
    with h5py.File(archive_path, 'r') as handle:
        components = {
            'lat': handle['lat'][:],
            'lon': handle['lon'][:],
            'pixel_rows': handle['pixel_rows'][:],
            'pixel_cols': handle['pixel_cols'][:],
            'support_mask': handle['support_mask'][:].astype(bool),
            'dates': pd.to_datetime([value.decode('utf-8') for value in handle['dates'][:]]),
            'S0_pred': handle['S0_pred'][:],
            'Sg_pred': handle['Sg_pred'][:],
            'S0_mean_map': handle['S0_mean_map'][:],
            'Sg_mean_map': handle['Sg_mean_map'][:],
            'S0_last_map': handle['S0_last_map'][:],
            'Sg_last_map': handle['Sg_last_map'][:],
        }
    return components


def reconstruct_prediction_map(values_2d, pixel_rows, pixel_cols, shape, time_index):
    full = np.full(shape, np.nan, dtype=np.float32)
    full[pixel_rows, pixel_cols] = values_2d[time_index].astype(np.float32)
    return full


def select_active_bbox(mask, margin_pixels=SUPPORT_MARGIN_PIXELS):
    rows, cols = np.where(mask)
    r0 = max(int(rows.min()) - margin_pixels, 0)
    r1 = min(int(rows.max()) + margin_pixels + 1, mask.shape[0])
    c0 = max(int(cols.min()) - margin_pixels, 0)
    c1 = min(int(cols.max()) + margin_pixels + 1, mask.shape[1])
    return r0, r1, c0, c1


def display_caption(text):
    display(Markdown(text))


def build_elastic_kernel(E, nu, dx, dy, a, nx, ny):
    xgrid = (np.arange(nx) - nx / 2) * dx
    ygrid = (np.arange(ny) - ny / 2) * dy
    xx, yy = np.meshgrid(xgrid, ygrid)
    r = np.sqrt(xx ** 2 + yy ** 2)
    r[r < 1e-6] = 1e-6
    return (1 + nu) / (np.pi * E * (1 - nu)) * (1 - np.exp(-r / a)) / r


def build_poroelastic_kernel(E, nu, alpha, hg, dx, dy, a, nx, ny):
    xgrid = (np.arange(nx) - nx / 2) * dx
    ygrid = (np.arange(ny) - ny / 2) * dy
    xx, yy = np.meshgrid(xgrid, ygrid)
    r = np.sqrt(xx ** 2 + yy ** 2)
    r[r < 1e-6] = 1e-6
    factor = alpha * (1 + nu) * hg * 9.81 / (np.pi * E * (1 - nu))
    return factor * (1 - np.exp(-r / a)) / r


def forward_two_layer_numpy(s0_tile, sg_tile, physics_cfg=PHYSICS_CFG):
    ny, nx = s0_tile.shape
    g_load = build_elastic_kernel(physics_cfg['E'], physics_cfg['nu'], physics_cfg['dx'], physics_cfg['dy'], physics_cfg['a_load'], nx, ny)
    g_poro = build_poroelastic_kernel(physics_cfg['E'], physics_cfg['nu'], physics_cfg['alpha'], physics_cfg['Hg'], physics_cfg['dx'], physics_cfg['dy'], physics_cfg['a_poro'], nx, ny)
    g_load_fft = np.fft.fft2(np.fft.ifftshift(g_load))
    g_poro_fft = np.fft.fft2(np.fft.ifftshift(g_poro))
    delta_l = physics_cfg['rho_w'] * s0_tile
    delta_p = physics_cfg['rho_w'] * physics_cfg['g'] * (sg_tile / physics_cfg['Seff'])
    uz_load = np.fft.ifft2(np.fft.fft2(delta_l) * g_load_fft).real
    uz_poro = np.fft.ifft2(np.fft.fft2(delta_p) * g_poro_fft).real
    return (uz_load + uz_poro).astype(np.float32)


def make_support_and_mask_figure(output_path):
    lat, lon, velocity = load_simple_h5_grid(VEL_PATH)
    _, _, coherence = load_simple_h5_grid(COH_PATH)
    _, _, time_valid_fraction = compute_time_valid_fraction(DISP_PATH)
    support_mask = build_support_mask(coherence, time_valid_fraction)
    extent = map_extent(lat, lon)

    velocity_masked = np.ma.masked_invalid(velocity)
    coherence_masked = np.ma.masked_invalid(coherence)
    time_valid_masked = np.ma.masked_invalid(time_valid_fraction)

    vel_vmin, vel_vmax = percentile_limits(velocity, symmetric=True, fallback=1.0)
    coh_vmin, coh_vmax = np.percentile(coherence[np.isfinite(coherence)], [5, 99])

    vel_cmap = plt.cm.RdBu_r.copy()
    vel_cmap.set_bad(color='#d9d9d9')
    coh_cmap = plt.cm.viridis.copy()
    coh_cmap.set_bad(color='#d9d9d9')
    frac_cmap = plt.cm.magma.copy()
    frac_cmap.set_bad(color='#d9d9d9')
    mask_cmap = plt.cm.Greys.copy()
    mask_cmap.set_bad(color='white')

    fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

    im0 = axes[0, 0].imshow(
        velocity_masked,
        origin='lower',
        extent=extent,
        cmap=vel_cmap,
        vmin=vel_vmin,
        vmax=vel_vmax,
        aspect='auto',
    )
    axes[0, 0].set_title('A. InSAR velocity')
    fig.colorbar(im0, ax=axes[0, 0], shrink=0.82, label=VELOCITY_UNIT_LABEL)

    im1 = axes[0, 1].imshow(
        coherence_masked,
        origin='lower',
        extent=extent,
        cmap=coh_cmap,
        vmin=float(coh_vmin),
        vmax=float(coh_vmax),
        aspect='auto',
    )
    axes[0, 1].set_title('B. Interferometric coherence')
    fig.colorbar(im1, ax=axes[0, 1], shrink=0.82, label=COHERENCE_UNIT_LABEL)

    im2 = axes[1, 0].imshow(
        time_valid_masked,
        origin='lower',
        extent=extent,
        cmap=frac_cmap,
        vmin=0.0,
        vmax=1.0,
        aspect='auto',
    )
    axes[1, 0].set_title('C. Temporal support fraction')
    fig.colorbar(im2, ax=axes[1, 0], shrink=0.82, label=FRACTION_UNIT_LABEL)

    im3 = axes[1, 1].imshow(
        np.ma.masked_where(~support_mask, support_mask.astype(float)),
        origin='lower',
        extent=extent,
        cmap=mask_cmap,
        vmin=0,
        vmax=1,
        aspect='auto',
    )
    axes[1, 1].set_title('D. Final support mask')
    fig.colorbar(im3, ax=axes[1, 1], shrink=0.82, ticks=[1], label='selected support')

    for ax in axes.ravel():
        ax.set_xlabel('Longitude [degrees_east]')
        ax.set_ylabel('Latitude [degrees_north]')

    finite_coh = coherence[np.isfinite(coherence)]
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return {
        'support_fraction': float(np.mean(support_mask)),
        'velocity_limits': [float(vel_vmin), float(vel_vmax)],
        'coherence_median': float(np.median(finite_coh)),
        'coherence_p75': float(np.percentile(finite_coh, 75)),
        'coherence_p90': float(np.percentile(finite_coh, 90)),
    }


def make_residual_gallery_figure(output_path, max_examples=3):
    archive = load_archive_components(ARCHIVE_PATH)
    all_dates = parse_acquisition_dates(DATE_PATH)
    date_to_archive_idx = {pd.Timestamp(date): idx for idx, date in enumerate(archive['dates'])}
    diag = json.load(open(EXPANDED_DIAG_JSON))
    example_specs = diag['tiles'][:max_examples]

    panels = []
    with h5py.File(DISP_PATH, 'r') as handle:
        disp = handle['z']
        for spec in example_specs:
            row = int(spec['tile_row'])
            col = int(spec['tile_col'])
            end_idx = int(spec['end_idx'])
            tile_slice = np.s_[row:row + TILE_SIZE, col:col + TILE_SIZE]
            raw_date = pd.Timestamp(all_dates[end_idx])
            archive_idx = date_to_archive_idx[raw_date]
            obs_tile = np.array(disp[end_idx, tile_slice[0], tile_slice[1]], dtype=np.float32)
            s0_map = reconstruct_prediction_map(archive['S0_pred'], archive['pixel_rows'], archive['pixel_cols'], archive['support_mask'].shape, archive_idx)
            sg_map = reconstruct_prediction_map(archive['Sg_pred'], archive['pixel_rows'], archive['pixel_cols'], archive['support_mask'].shape, archive_idx)
            pred_s0 = s0_map[tile_slice]
            pred_sg = sg_map[tile_slice]
            pred_disp = forward_two_layer_numpy(np.nan_to_num(pred_s0), np.nan_to_num(pred_sg))
            residual = pred_disp - np.nan_to_num(obs_tile)
            panels.append({
                'date': raw_date,
                'row': row,
                'col': col,
                'obs': obs_tile,
                'residual': residual,
                's0': pred_s0,
                'sg': pred_sg,
            })

    disp_stack = np.stack([np.nan_to_num(p['obs']) for p in panels])
    resid_stack = np.stack([p['residual'] for p in panels])
    s0_stack = np.stack([np.nan_to_num(p['s0']) for p in panels])
    sg_stack = np.stack([np.nan_to_num(p['sg']) for p in panels])
    disp_vmin, disp_vmax = percentile_limits(disp_stack, symmetric=True, fallback=1.0)
    resid_vmin, resid_vmax = percentile_limits(resid_stack, symmetric=True, fallback=1.0)
    s0_vmin, s0_vmax = percentile_limits(s0_stack, symmetric=True, fallback=1.0)
    sg_vmin, sg_vmax = percentile_limits(sg_stack, symmetric=True, fallback=1.0)

    fig, axes = plt.subplots(len(panels), 4, figsize=(15, 4.6 * len(panels)), constrained_layout=True)
    if len(panels) == 1:
        axes = np.expand_dims(axes, axis=0)

    disp_image = resid_image = s0_image = sg_image = None
    for row_idx, panel in enumerate(panels):
        row_title = f"date={panel['date'].date()} | row={panel['row']} | col={panel['col']}"
        data_panels = [
            ('Observed deformation', panel['obs'], 'coolwarm', disp_vmin, disp_vmax),
            ('Residual (pred - obs)', panel['residual'], 'coolwarm', resid_vmin, resid_vmax),
            ('Predicted $S_0$', panel['s0'], 'coolwarm', s0_vmin, s0_vmax),
            ('Predicted $S_g$', panel['sg'], 'coolwarm', sg_vmin, sg_vmax),
        ]
        for col_idx, (title, arr, cmap, vmin, vmax) in enumerate(data_panels):
            image = axes[row_idx, col_idx].imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
            if row_idx == 0:
                axes[row_idx, col_idx].set_title(title)
            if col_idx == 0:
                axes[row_idx, col_idx].set_ylabel(row_title)
            axes[row_idx, col_idx].set_xticks([])
            axes[row_idx, col_idx].set_yticks([])
            if col_idx == 0:
                disp_image = image
            elif col_idx == 1:
                resid_image = image
            elif col_idx == 2:
                s0_image = image
            else:
                sg_image = image

    cbar_kw = dict(orientation='horizontal', pad=0.08, fraction=0.05)
    fig.colorbar(disp_image, ax=axes[:, 0], **cbar_kw, label=DEFORMATION_UNIT_LABEL)
    fig.colorbar(resid_image, ax=axes[:, 1], **cbar_kw, label=DEFORMATION_UNIT_LABEL)
    fig.colorbar(s0_image, ax=axes[:, 2], **cbar_kw, label=STORAGE_UNIT_LABEL)
    fig.colorbar(sg_image, ax=axes[:, 3], **cbar_kw, label=STORAGE_UNIT_LABEL)

    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return pd.DataFrame(example_specs)


def make_prior_ablation_figure(output_path):
    summary = pd.DataFrame(json.load(open(PAPER_RESULTS_JSON))).copy()
    baseline = summary.loc[summary['label'] == 'Phase 1 Baseline'].iloc[0]
    summary['delta_val_forward_micro'] = 1e6 * (summary['val_forward'] - baseline['val_forward'])
    summary['delta_forward_rmse_norm_micro'] = 1e6 * (summary['forward_rmse_norm_mean'] - baseline['forward_rmse_norm_mean'])

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    x = np.arange(len(summary))
    colors = ['#2f4b7c'] + ['#bdbdbd'] * (len(summary) - 1)

    axes[0].bar(x, summary['delta_val_forward_micro'], color=colors)
    axes[0].axhline(0.0, color='black', linewidth=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(summary['label'], rotation=30, ha='right')
    axes[0].set_ylabel(r'$\Delta$ validation forward loss [$10^{-6}$ normalized deformation units$^2$]')
    axes[0].set_title('A. Change relative to frozen baseline')

    axes[1].bar(x, summary['delta_forward_rmse_norm_micro'], color=colors)
    axes[1].axhline(0.0, color='black', linewidth=0.8)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(summary['label'], rotation=30, ha='right')
    axes[1].set_ylabel(r'$\Delta$ validation forward RMSE [$10^{-6}$ normalized deformation units]')
    axes[1].set_title('B. Null-result prior ablation')

    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return summary[['label', 'val_forward', 'forward_rmse_norm_mean', 'delta_val_forward_micro', 'delta_forward_rmse_norm_micro']]


def make_baseline_export_panel(output_path):
    archive = load_archive_components(ARCHIVE_PATH)
    sample_pixels = pd.read_csv(SAMPLE_PIXELS_CSV)
    r0, r1, c0, c1 = select_active_bbox(archive['support_mask'])
    lat = archive['lat'][r0:r1]
    lon = archive['lon'][c0:c1]
    extent = map_extent(lat, lon)

    s0_mean = archive['S0_mean_map'][r0:r1, c0:c1]
    sg_mean = archive['Sg_mean_map'][r0:r1, c0:c1]
    s0_last = archive['S0_last_map'][r0:r1, c0:c1]
    sg_last = archive['Sg_last_map'][r0:r1, c0:c1]
    support_crop = archive['support_mask'][r0:r1, c0:c1]

    s0_vmin, s0_vmax = percentile_limits(np.stack([np.nan_to_num(s0_mean), np.nan_to_num(s0_last)]), symmetric=True, fallback=1.0)
    sg_vmin, sg_vmax = percentile_limits(np.stack([np.nan_to_num(sg_mean), np.nan_to_num(sg_last)]), symmetric=True, fallback=1.0)

    fig = plt.figure(figsize=(14, 12), constrained_layout=True)
    grid = fig.add_gridspec(3, 4)
    map_axes = [
        fig.add_subplot(grid[0, 0:2]),
        fig.add_subplot(grid[0, 2:4]),
        fig.add_subplot(grid[1, 0:2]),
        fig.add_subplot(grid[1, 2:4]),
    ]
    ts_axes = [
        fig.add_subplot(grid[2, 0]),
        fig.add_subplot(grid[2, 1]),
        fig.add_subplot(grid[2, 2]),
    ]
    legend_ax = fig.add_subplot(grid[2, 3])

    map_specs = [
        ('A. Mean predicted $S_0$', s0_mean, s0_vmin, s0_vmax),
        ('B. Mean predicted $S_g$', sg_mean, sg_vmin, sg_vmax),
        ('C. Last-date predicted $S_0$', s0_last, s0_vmin, s0_vmax),
        ('D. Last-date predicted $S_g$', sg_last, sg_vmin, sg_vmax),
    ]

    for ax, (title, arr, vmin, vmax) in zip(map_axes, map_specs):
        background = np.where(support_crop, 1.0, np.nan)
        ax.imshow(background, origin='lower', extent=extent, cmap='Greys', vmin=0, vmax=1, alpha=0.18, aspect='auto')
        image = ax.imshow(arr, origin='lower', extent=extent, cmap='coolwarm', vmin=vmin, vmax=vmax, aspect='auto')
        ax.set_title(title)
        ax.set_xlabel('Longitude [degrees_east]')
        ax.set_ylabel('Latitude [degrees_north]')
        fig.colorbar(image, ax=ax, shrink=0.80, label=STORAGE_UNIT_LABEL)

    pixel_lookup = {(int(r), int(c)): idx for idx, (r, c) in enumerate(zip(archive['pixel_rows'], archive['pixel_cols']))}
    for idx, row in sample_pixels.iterrows():
        color = f'C{idx}'
        for ax in map_axes:
            ax.plot(float(row['lon']), float(row['lat']), marker='o', markersize=4, color=color)
        pixel_idx = pixel_lookup[(int(row['row']), int(row['col']))]
        ts_ax = ts_axes[idx]
        ts_ax.plot(archive['dates'], archive['S0_pred'][:, pixel_idx], color='#1f77b4', label='$S_0$')
        ts_ax.plot(archive['dates'], archive['Sg_pred'][:, pixel_idx], color='#d62728', label='$S_g$')
        ts_ax.set_title(f"{chr(69 + idx)}. Pixel {idx + 1}")
        ts_ax.set_xlabel('Time')
        ts_ax.set_ylabel(STORAGE_UNIT_LABEL)
        ts_ax.tick_params(axis='x', rotation=30)
        ts_ax.text(
            0.03,
            0.97,
            f"lat={row['lat']:.3f}\nlon={row['lon']:.3f}",
            transform=ts_ax.transAxes,
            ha='left',
            va='top',
            fontsize=8,
            bbox={'facecolor': 'white', 'alpha': 0.7, 'edgecolor': 'none'},
        )
        ts_ax.grid(alpha=0.2)

    legend_ax.axis('off')
    legend_ax.plot([], [], color='#1f77b4', label='$S_0$')
    legend_ax.plot([], [], color='#d62728', label='$S_g$')
    legend_ax.legend(loc='center', frameon=False)
    legend_ax.set_title('Series legend')

    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return sample_pixels


def load_export_schema():
    with h5py.File(S0_NC_PATH, 'r') as s0_nc, h5py.File(SG_NC_PATH, 'r') as sg_nc:
        return {
            'archive_time_start': str(load_archive_components(ARCHIVE_PATH)['dates'][0].date()),
            'archive_time_end': str(load_archive_components(ARCHIVE_PATH)['dates'][-1].date()),
            'S0_nc_variables': list(s0_nc.keys()),
            'Sg_nc_variables': list(sg_nc.keys()),
        }



def load_synthetic_summary_table(summary_json_path, model_label, selected_noise=(0.0, 0.02, 0.05)):
    payload = json.load(open(summary_json_path))
    df = pd.DataFrame(payload['metrics'])
    keep = df[df['noise_scale'].isin(selected_noise)].copy()
    keep.insert(0, 'model', model_label)
    return keep[['model', 'noise_scale', 'layer', 'rmse', 'r2', 'corr', 'fit_slope', 'forward_residual_rmse']]


def build_synthetic_benchmark_tables():
    clean_df = load_synthetic_summary_table(SYN_CLEAN_SUMMARY_JSON, 'Dual-decoder clean anchor')
    hybrid_df = load_synthetic_summary_table(SYN_HYBRID_SUMMARY_JSON, 'Hybrid conditioned robustness branch')
    progression = pd.concat([clean_df, hybrid_df], ignore_index=True)
    multiseed = pd.read_csv(SYN_MULTISEED_SUMMARY_CSV)
    multiseed = multiseed[multiseed['noise_scale'].isin([0.0, 0.02, 0.05])].copy()
    return progression, multiseed


## Synthetic Experiment Results

This section pulls the synthetic model-selection and robustness results into the same notebook so the full paper figure set lives in one place.

In [ ]:
synthetic_progression_df, synthetic_multiseed_df = build_synthetic_benchmark_tables()
display_caption(
    "**Synthetic table summary.** The first table compares the clean dual-decoder anchor against the final hybrid conditioned robustness branch at selected noise levels. The second table reports the multi-seed mean and standard deviation for the best robustness branch."
)
display(synthetic_progression_df.round(4))
display(synthetic_multiseed_df.round(4))


## Figure 1: Synthetic Conditioning Problem

In [ ]:
display(Image(filename=str(SYN_FIG1_PATH)))
display_caption(
    "**Figure 1 caption.** Synthetic elastic-versus-poroelastic magnitude mismatch in the original two-signal setup. This figure motivates the balanced synthetic benchmark used for model development."
)


## Figure 2: Synthetic Architecture Progression

In [ ]:
display(Image(filename=str(SYN_FIG2_PATH)))
display_caption(
    "**Figure 2 caption.** Synthetic clean-case architecture progression. The dual-decoder frequency-separated model is the clean anchor selected from the synthetic benchmark ladder."
)


## Figure 3: Best Synthetic Robustness Branch

In [ ]:
display(Image(filename=str(SYN_FIG3_PATH)))
display_caption(
    "**Figure 3 caption.** Synthetic robustness-branch comparison. The hybrid direct-conditioned branch with noisy-stage $S_g$ emphasis is the best single-run robustness branch found during synthetic model selection."
)


## Figure 4: Synthetic Multi-Seed Validation

In [ ]:
display(Image(filename=str(SYN_FIG4_PATH)))
display_caption(
    "**Figure 4 caption.** Multi-seed validation of the best synthetic robustness branch. The clean case remains stable, whereas the noisy cases remain seed-sensitive, which is why the synthetic stage is reported as a strong proof of concept rather than a fully robust final solution."
)


## Export Schema Check

This cell confirms that the archive and NetCDF exports are the data sources used by the paper figures.


In [ ]:
schema = load_export_schema()
print(json.dumps(schema, indent=2))


## Figure 5: Punjab Data Support And Masking

In [ ]:
fig5_summary = make_support_and_mask_figure(FIG5_PATH)
display(Image(filename=str(FIG5_PATH)))
display_caption(
    f"**Figure 5 caption.** Punjab data support and masking for the real-data inversion. "
    f"Panel A shows the InSAR velocity field without the support overlay so the velocity texture remains readable. "
    f"Panel B shows coherence with a percentile-based stretch to make the low-to-moderate coherence structure visible. "
    f"Panel C shows the temporal-validity fraction computed from the full displacement stack, and Panel D shows the final support mask used for tile selection. "
    f"The coherence distribution is low to moderate overall (median {fig5_summary['coherence_median']:.3f}, 75th percentile {fig5_summary['coherence_p75']:.3f}, 90th percentile {fig5_summary['coherence_p90']:.3f}), and the selected support fraction is {fig5_summary['support_fraction']:.4f}."
)
fig5_summary


## Figure 6: Expanded Grouped-Support Baseline Residual Gallery

In [ ]:
fig6_examples = make_residual_gallery_figure(FIG6_PATH, max_examples=3)
display(Image(filename=str(FIG6_PATH)))
display_caption(
    "**Figure 6 caption.** Residual gallery for the frozen expanded grouped-support baseline. "
    "For each selected validation example, the panels show observed deformation, the residual after applying the forward model to the predicted states, and the corresponding predicted $S_0$ and $S_g$ fields. "
    f"Observed deformation and residual are shown in {DEFORMATION_UNIT_LABEL}. "
    f"The predicted storage fields are shown in {STORAGE_UNIT_LABEL}. "
    "The predicted deformation column was removed because the current baseline produces a very weak, smooth response that is more clearly communicated through the residual panel."
)
fig6_examples


## Figure 7: Punjab Baseline Versus Prior Ablations

In [ ]:
comparison_df = make_prior_ablation_figure(FIG7_PATH)
display(Image(filename=str(FIG7_PATH)))
display_caption(
    "**Figure 7 caption.** Null-result prior ablation for the real-data Punjab runs. "
    "The plotted values are changes relative to the frozen Phase 1 baseline. "
    f"Forward loss is reported in normalized deformation units squared, and normalized forward RMSE is reported in {NORMALIZED_DEFORMATION_UNIT_LABEL}. "
    "The point of this figure is not that one prior won; it is to show that the W3RA and GRACE formulations tested so far remained effectively tied with the baseline."
)
comparison_df


## Figure 8: Baseline Export Panel

In [ ]:
sample_pixels = make_baseline_export_panel(FIG8_PATH)
display(Image(filename=str(FIG8_PATH)))
display_caption(
    "**Figure 8 caption.** Baseline export panel from the frozen expanded grouped-support run. "
    "The map panels are cropped to the active supported domain so the predicted fields are visible rather than dominated by blank space. "
    f"All storage panels and time series are shown in {STORAGE_UNIT_LABEL}. "
    "These are relative latent-storage outputs from the generic-prior baseline, so they should be interpreted as internally consistent model fields rather than externally calibrated physical storage values."
)
sample_pixels


## Output Inventory

Running this notebook regenerates:
- `outputs/punjab_prior/figures/punjab_paper_figure5_support_mask_velocity.png`
- `outputs/punjab_prior/figures/punjab_paper_figure6_baseline_residual_gallery.png`
- `outputs/punjab_prior/figures/punjab_paper_figure7_prior_ablation_comparison.png`
- `outputs/punjab_prior/figures/punjab_paper_figure8_baseline_export_panel.png`
